In [ ]:
import pickle
from adaptive_latents import ArrayWithTime
import pandas
from adaptive_latents.utils import angle_between
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
def srs_to_l_df(srs):
    records = []
    for k, sr_list in srs.items():
        for sr_i, sr in enumerate(sr_list):
            latents: ArrayWithTime = sr.log['latents']
            for l_i, l in enumerate(sr.stim_designer.log):
                t_of_stim = l['time_of_stim']
                stim_sample = latents.time_to_sample(t_of_stim)
                old_v = latents[stim_sample-1] - latents[stim_sample-2]
                this_v = latents[stim_sample] - latents[stim_sample-1]
                l['old_v'] = old_v
                l['this_v'] = this_v

                records.append(dict(sr_key=k, sr_i=sr_i, l_i=l_i, l=l))
    return pandas.DataFrame(records)


with open("/mnt/data/al_cache/optim_open_vs_closed_077287296748976.pickle", 'rb') as f:
    df = srs_to_l_df(pickle.load(f))


In [ ]:
df.l[0].keys()

In [ ]:
df.l[0]['this_v'].shape

In [ ]:
# l = df.l[0]
# np.sign(np.linalg.det(np.squeeze([l['v'][:,0], l['s'], l['this_v']])))

In [ ]:
# df['theta'] = df['l'].apply(lambda l: angle_between(l['v'], l['observed_s_hat'])* np.pi/180)
# df['r'] = df['l'].apply(lambda l: np.linalg.norm(l['observed_s_hat']))

df['theta'] = df['l'].apply(lambda l: angle_between(l['v'], l['s'], radians=False) )
df['r'] = df['l'].apply(lambda l: np.linalg.norm(l['s']))



In [ ]:
%matplotlib qt
# fig, ax = plt.subplots(subplot_kw=dict(projection='polar'))
fig, axs = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, )



for k, ax in zip(df.sr_key.unique(), axs.flatten()):
    sub_df = df[df.sr_key == k]
    ax.scatter(sub_df['theta'], sub_df['r'], s=1, label=k, color='k')
    patch = plt.Rectangle(xy=(0,10), width=15, height=100, color='r', alpha=.1)
    ax.add_patch(patch)
    ax.set_xlim(xmin=0, xmax=180)
    ax.text(0.99, 0.97, k + f"\n {((sub_df.theta < 15) & (sub_df.r > 10)).sum()} / {len(sub_df)}", transform=ax.transAxes, ha='right', va='top')

    # ax.legend()
    ax.set_ylim(0,100)
    ax.axvline(90, linestyle='--', color='gray', alpha=0.5)

for ax in axs[-1,:]:
    ax.set_xlabel('Angle between v and s (degrees)')

for ax in axs[:,0]:
    ax.set_ylabel('Norm of s')